In [1]:
!pip install scrapy

  Using cached scrapy-2.19.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached aiohttp-3.14.5-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached backports_zstd-1.8.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (6.8 kB)
  Using cached brotli-1.2.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.1 kB)
  Using cached charset_normalizer-3.5.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (46 kB)
  Using cached cryptography-50.0.2-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (4.4 kB)
  Using cached cssselect-1.6.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached itemadapter-0.13.1-py3-none-any.whl.metadata (22 kB)
  Using cached itemloaders-1.5.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached lxml-6.1.3-cp313-cp313-manylinux_2_26_x86_64.manylin

In [2]:
%%writefile quotes_spider.py
import scrapy


class QuotesSpider(scrapy.Spider):
    name = "quotes"
    start_urls = [
        "https://quotes.toscrape.com/tag/humor/",
    ]

    def parse(self, response):
        for quote in response.css("div.quote"):
            yield {
                "author": quote.xpath("span/small/text()").get(),
                "text": quote.css("span.text::text").get(),
            }

        next_page = response.css('li.next a::attr("href")').get()
        if next_page is not None:
            yield response.follow(next_page, self.parse)

Writing quotes_spider.py


`scrapy runspider` runs the spider without creating a Scrapy project, and `-O` overwrites `quotes.jsonl` at every run (`-o` would append to the file).

**Politeness:** outside a project, Scrapy ignores `robots.txt` (`ROBOTSTXT_OBEY` is `False`) and doesn't space out its requests (`DOWNLOAD_DELAY` is `0`). The practice site `quotes.toscrape.com` is made to be crawled; on a real site, ask for both settings in the spider's `custom_settings` attribute:

```python
custom_settings = {"ROBOTSTXT_OBEY": True, "DOWNLOAD_DELAY": 1}
```

In [3]:
!scrapy runspider quotes_spider.py -O quotes.jsonl

2026-10-22 21:06:10 [scrapy.utils.log] INFO: Scrapy 2.19.0 started (bot: scrapybot)
2026-10-22 21:06:10 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.3',
 'libxml2': '2.14.6',
 'cssselect': '1.6.0',
 'parsel': '1.12.1',
 'w3lib': '2.5.0',
 'Twisted': '26.4.0',
 'Python': '3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ]',
 'pyOpenSSL': '26.4.0 (OpenSSL 4.0.3 29 Sep 2026)',
 'cryptography': '50.0.2',
 'Platform': 'Linux-6.12.111+deb13-amd64-x86_64-with-glibc2.41'}
2026-10-22 21:06:10 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-10-22 21:06:10 [asyncio] DEBUG: Using selector: EpollSelector
2026-10-22 21:06:10 [scrapy.addons] INFO: Enabled addons:
[]
2026-10-22 21:06:10 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-10-22 21:06:10 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-10-22 21:06:10 [scrapy.extensions.telnet] INFO: Telnet Password: 502a0cb21a70ce04
2026-10-

In [4]:
!cat quotes.jsonl

{"author": "Jane Austen", "text": "\u201cThe person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.\u201d"}
{"author": "Steve Martin", "text": "\u201cA day without sunshine is like, you know, night.\u201d"}
{"author": "Garrison Keillor", "text": "\u201cAnyone who thinks sitting in church can make you a Christian must also think that sitting in a garage can make you a car.\u201d"}
{"author": "Jim Henson", "text": "\u201cBeauty is in the eye of the beholder and it may be necessary from time to time to give a stupid or misinformed beholder a black eye.\u201d"}
{"author": "Charles M. Schulz", "text": "\u201cAll you need is love. But a little chocolate now and then doesn't hurt.\u201d"}
{"author": "Suzanne Collins", "text": "\u201cRemember, we're madly in love, so it's all right to kiss me anytime you feel like it.\u201d"}
{"author": "Charles Bukowski", "text": "\u201cSome people never go crazy. What truly horrible lives they must lead.\u201d"}
{"